# Buổi 5: Xử lý và biến đổi dữ liệu với Pandas
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Phát hiện và xử lý dữ liệu trùng lặp, thiếu, sai định dạng, ngoại lai
- Chuyển kiểu dữ liệu, xử lý ngày tháng, tạo cột mới
- Tổng hợp bằng groupby, pivot_table; kết hợp bảng bằng merge, concat
- Tính chỉ số RFM cho khách hàng

**Dữ liệu dùng trong buổi:** `ecommerce_orders_raw.csv`, `customers.csv` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [132]:
import numpy as np
import pandas as pd
DATA = "../data/"
pd.set_option("display.max_columns", 30)

## Phần A. Bài tập có hướng dẫn

Làm sạch bộ dữ liệu đơn hàng thương mại điện tử năm 2025 rồi phân tích.

### A1. Khám phá vấn đề của dữ liệu thô

In [133]:
raw = pd.read_csv(DATA + "ecommerce_orders_raw.csv")
print(raw.shape)
raw.info()
raw.sample(8, random_state=3)

(3816, 10)
<class 'pandas.DataFrame'>
RangeIndex: 3816 entries, 0 to 3815
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ma_don      3816 non-null   str    
 1   ngay_dat    3816 non-null   str    
 2   ma_kh       3816 non-null   str    
 3   vung        3796 non-null   str    
 4   kenh        3816 non-null   str    
 5   nhom_hang   3816 non-null   str    
 6   san_pham    3816 non-null   str    
 7   so_luong    3755 non-null   float64
 8   don_gia     3816 non-null   str    
 9   trang_thai  3816 non-null   str    
dtypes: float64(1), str(9)
memory usage: 298.3 KB


,ma_don,ngay_dat,ma_kh,vung,kenh,nhom_hang,san_pham,so_luong,don_gia,trang_thai
3087,DH001670,2025-05-22,KH0261,Miền Bắc,Website,Mỹ phẩm,Mỹ phẩm C,1.0,498000,Hoàn thành
454,DH002270,2025-02-13,KH0348,Miền Bắc,Sàn TMĐT,Mỹ phẩm,Mỹ phẩm C,1.0,461000,Hoàn thành
3255,DH003292,2025-07-24,KH0525,Miền Bắc,Cửa hàng,Điện thoại,Điện thoại C,2.0,4611000,Hoàn thành
1417,DH000787,2025-04-16,KH0122,Miền Nam,Sàn TMĐT,Phụ kiện,Phụ kiện D,1.0,493000,Đã hủy
6,DH000271,2025-09-06,KH0045,Miền Nam,App,Điện thoại,Điện thoại D,1.0,4919000,Hoàn thành
3397,DH002100,2025-06-13,KH0322,Miền Nam,Sàn TMĐT,Mỹ phẩm,Mỹ phẩm B,NaN,424000,Hoàn thành
2831,DH000274,17/06/2025,KH0045,Miền Nam,App,Thời trang,Thời trang B,3.0,386000,Hoàn thành
2340,DH002347,2025-07-06,KH0360,Miền Nam,App,Điện thoại,Điện thoại E,1.0,7333000,Hoàn thành


In [134]:
print("Giá trị thiếu mỗi cột:\n", raw.isna().sum())
print("\nSố dòng trùng hoàn toàn:", raw.duplicated().sum())
print("\nCác giá trị cột vùng:", raw["vung"].unique())
print("\nVí dụ đơn giá:", raw["don_gia"].sample(6, random_state=1).tolist())

Giá trị thiếu mỗi cột:
 ma_don         0
ngay_dat       0
ma_kh          0
vung          20
kenh           0
nhom_hang      0
san_pham       0
so_luong      61
don_gia        0
trang_thai     0
dtype: int64

Số dòng trùng hoàn toàn: 45

Các giá trị cột vùng: <StringArray>
[    'Miền Nam',   'Miền Trung',     'Miền Bắc',            nan,
   ' Miền Nam ',   ' Miền Bắc ', ' Miền Trung ',     'MIỀN NAM',
   'MIỀN TRUNG',     'MIỀN BẮC']
Length: 10, dtype: str

Ví dụ đơn giá: ['467000', '287000', '472000', '9109000', '8070000', '263000']


**Nhận xét:** (1) có dòng trùng; (2) `vung` viết không thống nhất và có giá trị thiếu; (3) `so_luong` thiếu;
(4) `don_gia` là chuỗi, có dạng "1.250.000 đ", có giá trị âm và giá trị bất thường; (5) `ngay_dat` lẫn hai định dạng.

### A2. Bỏ trùng lặp

In [135]:
df = raw.drop_duplicates().copy()
print("Trước:", len(raw), "→ Sau:", len(df))
print("Mã đơn còn trùng?", df["ma_don"].duplicated().sum())

Trước: 3816 → Sau: 3771
Mã đơn còn trùng? 0


### A3. Chuẩn hóa chuỗi và kiểu số

In [136]:
df["vung"] = df["vung"].str.strip().str.title()
print(df["vung"].value_counts(dropna=False))

def lam_sach_gia(x):
    if isinstance(x, str):
        x = x.replace("đ", "").replace(".", "").strip()
    return float(x)

df["don_gia"] = df["don_gia"].apply(lam_sach_gia)
df["so_luong"] = pd.to_numeric(df["so_luong"], errors="coerce")
df[["don_gia", "so_luong"]].describe()

vung
Miền Nam      1633
Miền Bắc      1344
Miền Trung     774
NaN             20
Name: count, dtype: int64


,don_gia,so_luong
count,3.771000e+03,3711.000000
mean,4.889065e+06,1.683374
std,3.668363e+07,0.745691
min,-1.000000e+00,1.000000
25%,3.830000e+05,1.000000
50%,5.400000e+05,2.000000
75%,5.633000e+06,2.000000
max,9.999990e+08,3.000000


### A4. Ngày tháng
`pd.to_datetime` với `format="mixed"` và `dayfirst=True` hiểu được cả `2025-03-15` lẫn `15/03/2025`.
Sau đó trích xuất tháng, quý, thứ trong tuần bằng `.dt`.

In [137]:
df["ngay_dat"] = pd.to_datetime(df["ngay_dat"], format="mixed", dayfirst=True)
df["thang"] = df["ngay_dat"].dt.month
df["quy"] = df["ngay_dat"].dt.quarter
df["thu"] = df["ngay_dat"].dt.day_name()
print(df["ngay_dat"].min(), "→", df["ngay_dat"].max())
df[["ngay_dat", "thang", "quy", "thu"]].head()

2025-01-01 00:00:00 → 2025-12-31 00:00:00


,ngay_dat,thang,quy,thu
0,2025-01-24,1,1,Friday
1,2025-12-18,12,4,Thursday
2,2025-05-06,5,2,Tuesday
3,2025-06-03,6,2,Tuesday
4,2025-01-26,1,1,Sunday


### A5. Giá trị thiếu và ngoại lai
Chiến lược phổ biến: **xóa** dòng (khi ít và không quan trọng), **điền** giá trị (trung vị, mode, giá trị từ nguồn khác),
hoặc **giữ** và đánh dấu. Ngoại lai phát hiện bằng quy tắc IQR: ngoài khoảng `[Q1 − 1,5·IQR; Q3 + 1,5·IQR]`
tính **trong từng nhóm hàng** (vì laptop vốn đắt hơn phụ kiện).

In [138]:
# 1) số lượng thiếu → điền bằng trung vị (1)
df["so_luong"] = df["so_luong"].fillna(df["so_luong"].median())

# 2) vùng thiếu → lấy từ bảng khách hàng
kh = pd.read_csv(DATA + "customers.csv")
vung_kh = kh.set_index("ma_kh")["vung"]
df["vung"] = df["vung"].fillna(df["ma_kh"].map(vung_kh))

# 3) đơn giá âm → lỗi nhập liệu, loại bỏ
df = df[df["don_gia"] > 0]

# 4) ngoại lai theo IQR trong từng nhóm hàng
def la_ngoai_lai(g):
    q1, q3 = g.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (g < q1 - 1.5 * iqr) | (g > q3 + 1.5 * iqr)

df["ngoai_lai"] = df.groupby("nhom_hang")["don_gia"].transform(la_ngoai_lai)
print("Số ngoại lai:", df["ngoai_lai"].sum())
print(df[df["ngoai_lai"]][["ma_don", "nhom_hang", "don_gia"]])
df = df[~df["ngoai_lai"]].drop(columns="ngoai_lai")
print("Còn lại:", len(df), "dòng; thiếu:", df.isna().sum().sum())

Số ngoại lai: 5
        ma_don   nhom_hang      don_gia
789   DH003663  Thời trang  999999000.0
863   DH001732    Phụ kiện  999999000.0
1635  DH003565  Điện thoại  999999000.0
2503  DH003296     Mỹ phẩm  999999000.0
3731  DH000032  Thời trang  999999000.0
Còn lại: 3761 dòng; thiếu: 0


### A6. Cột mới, lọc đơn hoàn thành và tổng hợp

In [139]:
df["so_luong"] = df["so_luong"].astype(int)
df["doanh_thu"] = df["so_luong"] * df["don_gia"]
ht = df[df["trang_thai"] == "Hoàn thành"].copy()
print("Tỷ lệ hủy đơn:", round((df["trang_thai"] == "Đã hủy").mean() * 100, 1), "%")

theo_thang = ht.groupby("thang").agg(so_don=("ma_don", "count"),
                                     doanh_thu_ty=("doanh_thu", lambda x: x.sum() / 1e9))
theo_thang.round(2)

Tỷ lệ hủy đơn: 19.8 %


,so_don,doanh_thu_ty
thang,,
1,343,1.76
2,216,1.23
3,235,1.47
4,270,1.71
5,238,1.30
6,226,1.38
7,227,1.44
8,254,1.32
9,228,1.52


`pivot_table` – bảng tổng hợp hai chiều giống PivotTable trong Excel:

In [140]:
pv = pd.pivot_table(ht, index="nhom_hang", columns="kenh", values="doanh_thu",
                    aggfunc="sum", margins=True, margins_name="Tổng") / 1e6
pv.round(0)

kenh,App,Cửa hàng,Sàn TMĐT,Website,Tổng
nhom_hang,,,,,
Gia dụng,183.0,90.0,156.0,170.0,599.0
Laptop,3206.0,1676.0,3100.0,2637.0,10619.0
Mỹ phẩm,107.0,46.0,101.0,77.0,330.0
Phụ kiện,140.0,66.0,142.0,122.0,470.0
Thời trang,134.0,60.0,147.0,125.0,466.0
Điện thoại,1438.0,840.0,1550.0,1606.0,5433.0
Tổng,5207.0,2779.0,5195.0,4736.0,17917.0


`merge` ghép thông tin khách hàng (tuổi, giới tính) vào đơn hàng:

In [141]:
ht = ht.merge(kh[["ma_kh", "tuoi", "gioi_tinh"]], on="ma_kh", how="left")
ht["nhom_tuoi"] = pd.cut(ht["tuoi"], bins=[17, 25, 35, 50, 65], labels=["18-25", "26-35", "36-50", "51-65"])
ht.groupby("nhom_tuoi", observed=True)["doanh_thu"].agg(["count", "mean"]).round(0)

,count,mean
nhom_tuoi,,
18-25,456,6433509.0
26-35,705,6150372.0
36-50,902,5978629.0
51-65,953,5513939.0


### A7. Phân tích RFM
RFM đánh giá giá trị khách hàng bằng 3 chỉ số: **R**ecency – số ngày từ lần mua gần nhất,
**F**requency – số đơn, **M**onetary – tổng chi tiêu. Mỗi chỉ số chấm điểm 1–5 bằng ngũ phân vị (`pd.qcut`).

In [142]:
ngay_moc = ht["ngay_dat"].max() + pd.Timedelta(days=1)
rfm = ht.groupby("ma_kh").agg(
    R=("ngay_dat", lambda x: (ngay_moc - x.max()).days),
    F=("ma_don", "nunique"),
    M=("doanh_thu", "sum"),
)
rfm["R_diem"] = pd.qcut(rfm["R"], 5, labels=[5, 4, 3, 2, 1]).astype(int)          # gần đây → điểm cao
rfm["F_diem"] = pd.qcut(rfm["F"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_diem"] = pd.qcut(rfm["M"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM"] = rfm["R_diem"].astype(str) + rfm["F_diem"].astype(str) + rfm["M_diem"].astype(str)
rfm.sort_values("M", ascending=False).head()

,R,F,M,R_diem,F_diem,M_diem,RFM
ma_kh,,,,,,,
KH0542,18,14,290436000.0,4,5,5,455
KH0475,38,16,239879000.0,3,5,5,355
KH0455,8,12,238994000.0,5,5,5,555
KH0513,15,11,233108000.0,5,5,5,555
KH0020,8,12,221724000.0,5,5,5,555


In [143]:
def phan_khuc(r):
    if r.R_diem >= 4 and r.F_diem >= 4:
        return "Khách hàng tốt nhất"
    if r.F_diem >= 4:
        return "Trung thành"
    if r.R_diem >= 4:
        return "Mới / tiềm năng"
    if r.R_diem <= 2 and r.F_diem >= 3:
        return "Có nguy cơ rời bỏ"
    if r.R_diem <= 2:
        return "Ngủ đông"
    return "Cần chăm sóc"

rfm["phan_khuc"] = rfm.apply(phan_khuc, axis=1)
tong_hop = rfm.groupby("phan_khuc").agg(so_kh=("R", "size"), R_tb=("R", "mean"), F_tb=("F", "mean"), M_tb_trieu=("M", lambda x: x.mean() / 1e6))
tong_hop.round(1).sort_values("M_tb_trieu", ascending=False)

,so_kh,R_tb,F_tb,M_tb_trieu
phan_khuc,,,,
Khách hàng tốt nhất,147,17.4,10.0,78.0
Trung thành,82,66.8,8.6,50.8
Có nguy cơ rời bỏ,54,175.9,3.6,10.3
Mới / tiềm năng,85,16.7,3.0,9.4
Cần chăm sóc,56,51.1,2.7,6.1
Ngủ đông,149,191.0,1.6,3.9


In [144]:
ht.to_csv("don_hang_sach.csv", index=False, encoding="utf-8-sig")
rfm.to_csv("rfm.csv", encoding="utf-8-sig")

## Phần B. Bài tập tự làm

Dùng các DataFrame `df` (đã làm sạch, gồm cả đơn hủy), `ht` (đơn hoàn thành) và `rfm` ở Phần A.

### Bài B1. Báo cáo chất lượng dữ liệu

Viết hàm `kiem_tra(df)` trả về DataFrame có một dòng cho mỗi cột với: kiểu dữ liệu, số giá trị thiếu, % thiếu,
số giá trị khác nhau. Chạy hàm trên `raw` và trên `df` để so sánh trước – sau khi làm sạch.

In [145]:
def kiem_tra(df):
   ket_qua = pd.DataFrame({
       "kieu_du_lieu": df.dtypes, #df.dtypes: lấy kiểu dữ liệu từng cột
       "so_gia_tri_thieu": df.isna().sum(), #df.isna().sum(): đếm số gtri thiếu
       "phan_tram_thieu": (df.isna().mean() * 100).round(2), #df.isna().mean()*100: tỷ lệ thiếu %
       "so_gia_tri_khac_nhau": df.nunique()
   })
   return ket_qua
#kiểm tra trước khi làm sạch
print("Chất lượng dữ liệu trước khi làm sạch")
kiem_tra(raw)

Chất lượng dữ liệu trước khi làm sạch


,kieu_du_lieu,so_gia_tri_thieu,phan_tram_thieu,so_gia_tri_khac_nhau
ma_don,str,0,0.00,3771
ngay_dat,str,0,0.00,511
ma_kh,str,0,0.00,600
vung,str,20,0.52,9
kenh,str,0,0.00,4
nhom_hang,str,0,0.00,6
san_pham,str,0,0.00,30
so_luong,float64,61,1.60,3
don_gia,str,0,0.00,1874
trang_thai,str,0,0.00,2


In [146]:
#kiểm tra sau khi làm sạch
print("Chất lượng dữ liệu sau khi làm sạch")
kiem_tra(df)
#trả về giá trị thiếu bằng 0 vì câu lệnh ở phần A
#df["so_luong"] = df["so_luong"].fillna(df["so_luong"].median())
# fillna() dùng để thay thế giá trị thiếu.
# median() tính trung vị của cột so_luong.
# Những ô bị thiếu được thay bằng giá trị trung vị.

Chất lượng dữ liệu sau khi làm sạch


,kieu_du_lieu,so_gia_tri_thieu,phan_tram_thieu,so_gia_tri_khac_nhau
ma_don,str,0,0.0,3761
ngay_dat,datetime64[us],0,0.0,365
ma_kh,str,0,0.0,600
vung,str,0,0.0,3
kenh,str,0,0.0,4
nhom_hang,str,0,0.0,6
san_pham,str,0,0.0,30
so_luong,int64,0,0.0,3
don_gia,float64,0,0.0,1810
trang_thai,str,0,0.0,2


### Bài B2. Doanh thu theo vùng và quý

Từ `ht`, lập pivot table doanh thu (triệu đồng) với dòng là vùng, cột là quý. Thêm cột tăng trưởng quý 4 so với quý 3 (%).
Vùng nào tăng trưởng mạnh nhất?

In [147]:
# pivot table tổng doanh thu với dòng là vùng, cột là quý
pv_vung_quy = pd.pivot_table(ht,
                             index="vung",
                             columns="quy",
                             values="doanh_thu",
                             aggfunc="sum") /1e6
# thêm cột tính tỷ lệ tăng trưởng quý 4 so với quý 3
pv_vung_quy["tang_truong_Q4_Q3"] = ((pv_vung_quy[4] - pv_vung_quy[3])/pv_vung_quy[3] * 100).round(1)
print("doanh thu theo vùng và quý (triệu đồng)")
display(pv_vung_quy)
# vùng tăng trưởng mạnh nhất
vung_max = pv_vung_quy["tang_truong_Q4_Q3"].idxmax()
muc_max = pv_vung_quy["tang_truong_Q4_Q3"].max()
print("vùng có mức tăng trưởng doanh thu mạnh nhất Q4:", vung_max, ", tỷ lệ tăng trưởng", muc_max, "%")

doanh thu theo vùng và quý (triệu đồng)


quy,1,2,3,4,tang_truong_Q4_Q3
vung,,,,,
Miền Bắc,1511.817,1492.217,1620.803,1605.229,-1.0
Miền Nam,2010.296,2108.856,1782.308,1917.999,7.6
Miền Trung,940.444,792.266,880.264,1254.700,42.5


vùng có mức tăng trưởng doanh thu mạnh nhất Q4: Miền Trung , tỷ lệ tăng trưởng 42.5 %


### Bài B3. Tỷ lệ hủy đơn theo kênh

Từ `df` (gồm cả đơn hủy), tính tỷ lệ hủy đơn (%) theo từng kênh bán và theo nhóm hàng.
Kênh/nhóm hàng nào có tỷ lệ hủy cao nhất? Gợi ý một hành động cho quản lý.

<details><summary>💡 Gợi ý</summary>

Tạo cột 0/1 cho đơn hủy rồi lấy trung bình theo nhóm = tỷ lệ.

</details>

In [148]:
#cột 0/1 đánh dấu đơn huỷ
df["la_don_huy"] = (df["trang_thai"] == "Đã hủy").astype(int)
#tỷ lệ huỷ theo kênh bán
huy_theo_kenh = (df.groupby("kenh")["la_don_huy"].mean()*100).round(1).sort_values(ascending=False)
print("Tỷ lệ huỷ đơn hàng theo kênh (%)")
display(huy_theo_kenh.to_frame(name="Tỷ lệ hủy"))
#huỷ theo nhóm hàng
huy_theo_nhom = (df.groupby("nhom_hang")["la_don_huy"].mean()*100).round(1).sort_values(ascending=False)
print("Tỷ lệ huỷ đơn theo nhóm hàng")
display(huy_theo_nhom.to_frame(name="Tỷ lệ huỷ"))
print("Kênh có tỷ lệ huỷ cao nhất:", huy_theo_kenh.idxmax(), "tỷ lệ huỷ(%)", huy_theo_kenh.max())
print("Nhóm hàng có tỷ lệ huỷ cao nhất:", huy_theo_nhom.idxmax(), "tỷ lệ huỷ(%)", huy_theo_nhom.max())

Tỷ lệ huỷ đơn hàng theo kênh (%)


,Tỷ lệ hủy
kenh,
App,21.1
Cửa hàng,19.4
Sàn TMĐT,19.2
Website,19.2


Tỷ lệ huỷ đơn theo nhóm hàng


,Tỷ lệ huỷ
nhom_hang,
Laptop,21.2
Điện thoại,20.9
Thời trang,19.5
Phụ kiện,19.3
Gia dụng,19.2
Mỹ phẩm,18.9


Kênh có tỷ lệ huỷ cao nhất: App tỷ lệ huỷ(%) 21.1
Nhóm hàng có tỷ lệ huỷ cao nhất: Laptop tỷ lệ huỷ(%) 21.2


### Bài B4. Top sản phẩm và giỏ hàng trung bình

(a) 10 sản phẩm có doanh thu cao nhất và tỷ trọng cộng dồn (%) của chúng (gợi ý `cumsum`).
(b) Giá trị đơn hàng trung bình (AOV) theo kênh và theo tháng; tháng nào AOV cao nhất?

In [149]:
top_sp = ht.groupby("san_pham")["doanh_thu"].sum().sort_values(ascending=False).head(10).to_frame()
#tỷ trọng doanh thu theo sản phẩn
tong_doanh_thu = ht["doanh_thu"].sum()
top_sp["ty_trong"] = (top_sp["doanh_thu"] / tong_doanh_thu * 100).round(2)
#tỷ trọng cộng dồn (%)
top_sp["ty_trong_cong_don"] = top_sp["ty_trong"].cumsum().round(2) #cumsum:tỷ trọng cộng dồn
print("Top 10 sản phẩm doanh thu cao nhất")
display(top_sp)

Top 10 sản phẩm doanh thu cao nhất


,doanh_thu,ty_trong,ty_trong_cong_don
san_pham,,,
Laptop C,2.397842e+09,13.38,13.38
Laptop D,2.177400e+09,12.15,25.53
Laptop B,2.138361e+09,11.93,37.46
Laptop E,1.997867e+09,11.15,48.61
Laptop A,1.907419e+09,10.65,59.26
Điện thoại C,1.237933e+09,6.91,66.17
Điện thoại D,1.136647e+09,6.34,72.51
Điện thoại E,1.069865e+09,5.97,78.48
Điện thoại B,1.007986e+09,5.63,84.11


In [150]:
#Giá trọ đơn hàng trung bình (AOV)
aov_kenh_thang = ht.groupby(["thang", "kenh"])["doanh_thu"].mean().unstack()
print("aov theo kênh và tháng (vnđ)")
display(aov_kenh_thang.round(1))
aov_chung = ht.groupby("thang")["doanh_thu"].mean()
thang_max = aov_chung.idxmax()
print(f"Tháng {thang_max} có giá trị đơn hàng trung bình (AOV) cao nhất toàn sàn: {aov_chung.max():,.0f} vnđ/đơn")

aov theo kênh và tháng (vnđ)


kenh,App,Cửa hàng,Sàn TMĐT,Website
thang,,,,
1,4625227.3,6652069.8,6275508.1,3256079.5
2,4639895.5,7046739.1,7341892.3,4620032.8
3,8658507.9,4553950.0,6016616.4,5185593.2
4,6091577.8,6175689.7,5534452.4,7772761.2
5,5768392.4,7131500.0,3802274.2,5894188.4
6,4003955.2,6549525.0,6046152.5,8258983.3
7,8266542.4,6426562.5,6800706.7,3901704.9
8,6268904.1,5433023.3,3302253.7,5775774.6
9,5027671.1,6052400.0,5933260.9,9940569.0


Tháng 11 có giá trị đơn hàng trung bình (AOV) cao nhất toàn sàn: 7,082,132 vnđ/đơn


### Bài B5. Nối dữ liệu RFM với thông tin khách hàng

Ghép `rfm` với `customers.csv` (dùng `merge`). Cho biết với mỗi phân khúc: số khách, tuổi trung bình, tỷ lệ nữ (%)
và vùng có nhiều khách nhất. Khách hàng có nguy cơ rời bỏ tập trung ở vùng nào?

In [151]:
rfm_kh = rfm.reset_index().merge(kh, on="ma_kh", how="left")
rfm_kh["la_nu"] = (rfm_kh["gioi_tinh"] == "Nữ").astype(int)
bc_phan_khuc = rfm_kh.groupby("phan_khuc").agg(
    so_khach=("ma_kh", "count"),
    tuoi_tb=("tuoi", "mean"),
    ty_le_nu=("la_nu", lambda x: x.mean() * 100),
    vung_tap_trung=("vung", lambda x: x.mode()[0] if not x.mode().empty else None)
).round(1)
print("Hồ sơ khách hàng theo phân khúc rfm")
display(bc_phan_khuc)
vung_nguy_co = bc_phan_khuc.loc["Có nguy cơ rời bỏ", "vung_tap_trung"]
print(f"Tập khách hàng có nguy cơ rời bỏ nhiều nhất ở: {vung_nguy_co}.")

Hồ sơ khách hàng theo phân khúc rfm


,so_khach,tuoi_tb,ty_le_nu,vung_tap_trung
phan_khuc,,,,
Có nguy cơ rời bỏ,54,39.7,42.6,Miền Nam
Cần chăm sóc,56,39.7,55.4,Miền Bắc
Khách hàng tốt nhất,147,41.2,52.4,Miền Nam
Mới / tiềm năng,85,41.9,41.2,Miền Nam
Ngủ đông,149,42.0,53.7,Miền Bắc
Trung thành,82,43.2,46.3,Miền Nam


Tập khách hàng có nguy cơ rời bỏ nhiều nhất ở: Miền Nam.


## Phần C. Bài tập làm thêm

### Bài C1. Phân tích nhóm khách hàng theo tháng mua đầu tiên (cohort)

Xác định tháng mua đầu tiên của mỗi khách hàng. Lập bảng cohort: dòng = tháng mua đầu, cột = số tháng kể từ lần đầu
(0, 1, 2, ...), giá trị = tỷ lệ khách còn quay lại mua (%). Nhận xét tỷ lệ giữ chân sau 3 tháng.

In [152]:
# Viết code của bạn ở đây


### Bài C2. Doanh thu theo ngày trong tuần và trung bình trượt

(a) Doanh thu trung bình theo thứ trong tuần (sắp xếp Thứ 2 → Chủ nhật, đổi tên sang tiếng Việt).
(b) Tạo chuỗi doanh thu theo ngày (`resample("D")`), tính trung bình trượt 7 ngày và tìm 3 tuần có doanh thu cao nhất.

In [153]:
# Viết code của bạn ở đây


### Bài C3. Đóng gói quy trình làm sạch

Viết hàm `lam_sach(duong_dan_raw, duong_dan_kh)` gom toàn bộ các bước A2–A6 thành một hàm, trả về DataFrame sạch.
Kiểm tra: kết quả của hàm phải có cùng số dòng với `df` ở Phần A. Vì sao đóng gói thành hàm lại quan trọng
khi dữ liệu được cập nhật hằng tuần?

In [154]:
# Viết code của bạn ở đây
